In [89]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
from models import ContrastiveSpaceModel
import pickle
import numpy as np
import os
from train_utils import train_contrastive
from data_utils import ContrastiveCollator

In [90]:
source_model = 'graph'
source_key = f'{source_model}_embeddings'
batch_size = 4
device_name = 'cuda:0'
shared_dim = 64

val_path = 'data/contrastive_dataset/CA_test.pickle'

with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

collator = ContrastiveCollator(pad_id=0)

source_dim = val_dataset[0][source_key].shape[0]
transformer_dim = val_dataset[0]['transformer_embeddings'].shape[0]

In [91]:
print(len(val_dataset))

756


In [92]:
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [93]:
if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

model = ContrastiveSpaceModel(source_dim, transformer_dim, shared_dim=shared_dim)
model.to(device)

checkpoint = torch.load(f'saved_models/contrastive/{source_model}.pt', map_location=device_name)
model.load_state_dict(checkpoint)
model.eval()

ContrastiveSpaceModel(
  (source_proj): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=64, bias=True)
    )
  )
  (transformer_proj): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=64, bias=True)
    )
  )
)

In [94]:
def collect_embeddings(model, dataloader, source_key, device):
    model.eval()
    
    all_z_s = []
    all_z_t = []

    with torch.no_grad():
        for batch in dataloader:
            source_emb = batch[source_key].to(device)
            transformer_emb = batch['transformer_embeddings'].to(device)

            z_s, z_t, temp = model(source_emb, transformer_emb)

            all_z_s.append(z_s)
            all_z_t.append(z_t)

    Zs = torch.cat(all_z_s, dim=0)
    Zt = torch.cat(all_z_t, dim=0)

    return Zs, Zt

In [95]:
Zs, Zt = collect_embeddings(model, val_loader, source_key, device)

In [96]:
print(Zs.shape, Zt.shape)

torch.Size([756, 64]) torch.Size([756, 64])


In [97]:
def procrustes_error(Zs, Zt):
    """
    Computes orthogonal Procrustes alignment error.
    Returns Frobenius norm error and aligned embeddings.
    """

    # Center both spaces
    Zs_centered = Zs - Zs.mean(dim=0, keepdim=True)
    Zt_centered = Zt - Zt.mean(dim=0, keepdim=True)

    # Compute cross-covariance
    M = Zs_centered.T @ Zt_centered

    # SVD
    U, _, Vt = torch.linalg.svd(M)
    R = U @ Vt

    # Align Zs
    Zs_aligned = Zs_centered @ R

    # Frobenius norm error
    error = torch.norm(Zs_aligned - Zt_centered, p='fro')

    return error.item(), Zs_aligned, Zt_centered

In [98]:
proc_error, Zs_aligned, Zt_centered = procrustes_error(Zs, Zt)
print("Procrustes error:", proc_error)

Procrustes error: 17.57312774658203


In [99]:
import torch.nn.functional as F

def distance_matrix(Z):
    return torch.cdist(Z, Z, p=2)

def distance_matrix_correlation(Zs, Zt):
    Ds = distance_matrix(Zs)
    Dt = distance_matrix(Zt)

    # Flatten upper triangular (without diagonal)
    idx = torch.triu_indices(Ds.size(0), Ds.size(1), offset=1)

    ds_flat = Ds[idx[0], idx[1]]
    dt_flat = Dt[idx[0], idx[1]]

    # Pearson correlation
    corr = torch.corrcoef(torch.stack([ds_flat, dt_flat]))[0,1]

    return corr.item()

In [100]:
rsa_corr = distance_matrix_correlation(Zs, Zt)
print("Distance matrix correlation:", rsa_corr)

Distance matrix correlation: 0.7262511253356934


In [101]:
def retrieval_accuracy(Zs, Zt, topk=(1,5)):
    """
    Computes cross-space retrieval accuracy.
    """

    # Normalize (important!)
    Zs = F.normalize(Zs, dim=1)
    Zt = F.normalize(Zt, dim=1)

    # Similarity matrix
    sim = Zt @ Zs.T   # [N, N]

    results = {}

    for k in topk:
        correct = 0
        topk_indices = sim.topk(k, dim=1).indices

        for i in range(sim.size(0)):
            if i in topk_indices[i]:
                correct += 1

        results[f"top{k}"] = correct / sim.size(0)

    return results

In [102]:
retrieval = retrieval_accuracy(Zs, Zt)
print(retrieval)

{'top1': 0.10978835978835978, 'top5': 0.3306878306878307}


In [103]:
def evaluate_alignment(model, dataloader, source_key, device):

    Zs, Zt = collect_embeddings(model, dataloader, source_key, device)

    proc_error, _, _ = procrustes_error(Zs, Zt)
    rsa_corr = distance_matrix_correlation(Zs, Zt)
    retrieval = retrieval_accuracy(Zs, Zt)

    print("=== Alignment Evaluation ===")
    print("Procrustes error:", proc_error)
    print("Distance matrix correlation:", rsa_corr)
    print("Retrieval accuracy:", retrieval)

    return {
        "procrustes_error": proc_error,
        "rsa_correlation": rsa_corr,
        "retrieval": retrieval
    }

In [104]:
evaluate_alignment(model, val_loader, source_key, device)

=== Alignment Evaluation ===
Procrustes error: 17.57312774658203
Distance matrix correlation: 0.7262511253356934
Retrieval accuracy: {'top1': 0.10978835978835978, 'top5': 0.3306878306878307}


{'procrustes_error': 17.57312774658203,
 'rsa_correlation': 0.7262511253356934,
 'retrieval': {'top1': 0.10978835978835978, 'top5': 0.3306878306878307}}